# Fine-tune YOLO11n from `best.pt` (4-class) — **Kaggle**

Combines DataMatrix + EAN13/CODE128/GS1_128 and fine-tunes from existing `best.pt`.

## Target classes
- `0: datamatrix`
- `1: EAN13`
- `2: CODE128`
- `3: GS1_128`

## Why retrain again?
The first 4-class export localizes **DataMatrix well**, but on pharmacy/warehouse photos **EAN13 and GS1_128 scores are ~0** (CODE128 weak). App conf/NMS cannot fix that — need more 1D/pharma labels and a longer balanced fine-tune. After train, check `Box counts by class` — if EAN13/GS1 are tiny vs DM, expect the same failure.

**Optional faster path for locate-only:** collapse EAN13+CODE128+GS1_128 → one class `barcode_1d` (2-class model). Better recall; ZXing still decides format.

## Setup
1. Dataset `barcode-yolo-4class-train` should contain:
   - `best.pt`
   - `barcode_best_training_images/` **or** `.zip`
   - `barcode_dataset_merged/` **or** `.zip`
2. Session options: **GPU T4**, Internet **On**
3. **+ Add Input** → attach dataset
4. Run top → bottom

> Kaggle often auto-extracts zips into folders. This notebook supports **both**.


## 1) GPU check + install


In [ ]:
!nvidia-smi
!pip -q install -U ultralytics

import torch
import ultralytics
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Enable GPU: Session options → Accelerator → GPU T4, then Restart session"


## 2) Stage dataset (folders or zips) into `/kaggle/working`


In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

print("Inputs:")
!ls -la /kaggle/input

hits = list(INPUT_ROOT.rglob("best.pt"))
assert hits, "best.pt not found under /kaggle/input — Add Input dataset first"
DATASET_DIR = hits[0].parent
print("Using dataset dir:", DATASET_DIR)
!ls -lah {DATASET_DIR}

shutil.copy2(DATASET_DIR / "best.pt", WORK / "best.pt")
print("Copied best.pt")


def find_asset(name_stem: str):
    z = DATASET_DIR / f"{name_stem}.zip"
    d = DATASET_DIR / name_stem
    if z.exists():
        return "zip", z
    if d.exists() and d.is_dir():
        return "dir", d
    zips = list(INPUT_ROOT.rglob(f"{name_stem}.zip"))
    if zips:
        return "zip", zips[0]
    dirs = [p for p in INPUT_ROOT.rglob(name_stem) if p.is_dir()]
    if dirs:
        return "dir", dirs[0]
    return None, None


def stage_dataset(name_stem: str, dest: Path):
    kind, src = find_asset(name_stem)
    assert src is not None, f"Missing {name_stem} (.zip or folder) under /kaggle/input"
    print(f"{name_stem}: {kind} -> {src}")
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    if kind == "zip":
        with zipfile.ZipFile(src, "r") as zf:
            zf.extractall(dest)
    else:
        shutil.copytree(src, dest, dirs_exist_ok=True)
    return dest


dm_root = stage_dataset("barcode_best_training_images", WORK / "dm_ds")
bc_root = stage_dataset("barcode_dataset_merged", WORK / "bc_ds")

def unwrap_if_nested(root: Path) -> Path:
    if (root / "images").exists():
        return root
    nested = list(root.rglob("images/train"))
    if nested:
        return nested[0].parent.parent
    nested2 = list(root.rglob("images"))
    for p in nested2:
        if (p / "train").exists():
            return p.parent
    return root

dm_root = unwrap_if_nested(dm_root)
bc_root = unwrap_if_nested(bc_root)

print("DM root:", dm_root, "images?", (dm_root / "images").exists())
print("BC root:", bc_root, "images?", (bc_root / "images").exists())
assert (dm_root / "images").exists(), f"DM images not found under {dm_root}"
assert (bc_root / "images").exists(), f"BC images not found under {bc_root}"

Path("/kaggle/working/DM_ROOT.txt").write_text(str(dm_root))
Path("/kaggle/working/BC_ROOT.txt").write_text(str(bc_root))
print("OK — staged datasets")


## 3) Build combined 4-class dataset


In [ ]:
import shutil
from collections import Counter
from pathlib import Path

dm_root = Path(Path("/kaggle/working/DM_ROOT.txt").read_text().strip())
bc_root = Path(Path("/kaggle/working/BC_ROOT.txt").read_text().strip())

combined = Path("/kaggle/working/combined_4class")
if combined.exists():
    shutil.rmtree(combined)

for split in ["train", "val", "test"]:
    (combined / "images" / split).mkdir(parents=True, exist_ok=True)
    (combined / "labels" / split).mkdir(parents=True, exist_ok=True)

valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
NL = chr(10)

def copy_pair(src_img, src_lbl, dst_img_dir, dst_lbl_dir, class_shift, out_stem=None):
    if not src_img.exists() or not src_lbl.exists():
        return False
    stem = out_stem or src_img.stem
    dst_img = dst_img_dir / f"{stem}{src_img.suffix.lower()}"
    dst_lbl = dst_lbl_dir / f"{stem}.txt"
    shutil.copy2(src_img, dst_img)
    out_lines = []
    for line in src_lbl.read_text().strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cid = int(float(parts[0])) + class_shift
        out_lines.append(" ".join([str(cid)] + parts[1:]))
    dst_lbl.write_text(NL.join(out_lines) + (NL if out_lines else ""))
    return True

dm_counts = Counter()
for split in ["train", "val", "test"]:
    dm_img = dm_root / "images" / split
    dm_lbl = dm_root / "labels" / split
    if not dm_img.exists() or not dm_lbl.exists():
        continue
    for img in sorted(dm_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        lbl = dm_lbl / f"{img.stem}.txt"
        if copy_pair(img, lbl, combined / "images" / split, combined / "labels" / split, 0, out_stem=f"dm_{img.stem}"):
            dm_counts[split] += 1

bc_counts = Counter()
for split in ["train", "val", "test"]:
    bc_img = bc_root / "images" / split
    bc_lbl = bc_root / "labels" / split
    if not bc_img.exists() or not bc_lbl.exists():
        continue
    for img in sorted(bc_img.iterdir()):
        if img.suffix.lower() not in valid_ext:
            continue
        lbl = bc_lbl / f"{img.stem}.txt"
        if copy_pair(img, lbl, combined / "images" / split, combined / "labels" / split, 1, out_stem=f"bc_{img.stem}"):
            bc_counts[split] += 1

print("DM image counts:", dict(dm_counts))
print("BC image counts:", dict(bc_counts))
Path("/kaggle/working/COMBINED_ROOT.txt").write_text(str(combined))
print("Combined:", combined)


## 4) Write `data.yaml` + sanity counts


In [ ]:
from pathlib import Path
from collections import Counter

combined = Path(Path("/kaggle/working/COMBINED_ROOT.txt").read_text().strip())

yaml_text = """path: /kaggle/working/combined_4class
train: images/train
val: images/val
test: images/test

names:
  0: datamatrix
  1: EAN13
  2: CODE128
  3: GS1_128
"""
(combined / "data.yaml").write_text(yaml_text)
print((combined / "data.yaml").read_text())

img_counts = {}
box_counts = Counter()
for split in ["train", "val", "test"]:
    imgs = [p for p in (combined / "images" / split).iterdir() if p.is_file()]
    lbls = list((combined / "labels" / split).glob("*.txt"))
    img_counts[split] = len(imgs)
    for lf in lbls:
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                box_counts[int(float(line.split()[0]))] += 1

print("Image counts:", img_counts)
print("Box counts by class:", dict(sorted(box_counts.items())))
assert sum(img_counts.values()) > 0
assert img_counts.get("val", 0) > 0, "val split empty — check source datasets"


## 5) Fine-tune from `best.pt`


In [ ]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/kaggle/working/best.pt")
DATA_YAML = Path("/kaggle/working/combined_4class/data.yaml")
assert BEST_PT.exists() and DATA_YAML.exists()

PROJECT = "/kaggle/working/runs/barcode"
RUN_NAME = "yolo11n_960_dm_ean_code128_gs1_v2"
IMGSZ = 960
EPOCHS = 150
BATCH = 8  # drop to 4 if OOM
# Longer fine-tune + freeze early so rare EAN/GS1 heads can move without wiping DM.
# Set freeze=0 if 1D still dead after this run.

model = YOLO(str(BEST_PT))
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    patience=40,
    freeze=10,
    cls=1.0,
    box=7.5,
    close_mosaic=20,
    cos_lr=True,
)
print("Done:", Path(PROJECT) / RUN_NAME / "weights")


## 6) Validate


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/kaggle/working/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt"
BEST = candidates[0]
print("Using", BEST)

model = YOLO(str(BEST))
metrics = model.val(data="/kaggle/working/combined_4class/data.yaml", imgsz=960)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))


## 7) Export ONNX


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

PROJECT = Path("/kaggle/working/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
out = model.export(format="onnx", imgsz=960, simplify=True)
print("ONNX:", out)

onnx_dst = Path("/kaggle/working/barcode-yolo11n-4class.onnx")
shutil.copy2(Path(out), onnx_dst)
shutil.copy2(BEST, "/kaggle/working/best_4class.pt")
print("Wrote", onnx_dst)
print("Wrote /kaggle/working/best_4class.pt")


## 8) Download outputs

Use Kaggle Output panel or FileLink below.

> App currently expects single-class ONNX (`1×5×N`). 4-class model needs parser update before replacing production ONNX.


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

for p in [
    Path("/kaggle/working/best_4class.pt"),
    Path("/kaggle/working/barcode-yolo11n-4class.onnx"),
]:
    assert p.exists(), p
    print(p, round(p.stat().st_size / (1024 * 1024), 2), "MB")
    display(FileLink(str(p)))


## Notes
- Your layout (`best.pt` + 2 folders) is supported.
- **OOM:** `BATCH = 4` or smoke at `imgsz=640`, then final at 960.
- **Session options** (right panel) → GPU + Internet; **Save Version** after training.
- After combine, check **Box counts by class**. If EAN13/GS1 ≪ DM+CODE128, those heads stay near zero — add labels or train 2-class (`datamatrix` + `barcode_1d`).
- App probe after export: `python3 scripts/probe-yolo-4class.py your.png` — want EAN/GS1 max ≫ 0.1 (not ~0.001).
